In [1]:
from pathlib import Path
import numpy as np
import torch
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, Sampler
import random
from itertools import islice

ROOT = Path("../data/processed/classification").resolve()

transforms = v2.Compose([
  v2.ToImage(), 
  v2.Resize((64,64)),
])
train_ds = ImageFolder(ROOT / "train", transform=transforms)
dev_ds = ImageFolder(ROOT / "dev", transform=transforms)

print(len(dev_ds.targets))
print(dev_ds.targets[:10])


104016
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
class BalancedBatchSampler(Sampler):
  def __init__(self, targets, batch_size=12, g=torch.Generator(), shuffle=False):

    self.targets = np.array(targets, dtype=np.int64)
    self.batch_size = batch_size
    self.classes = np.unique(self.targets).tolist()
    self.n_classes = len(self.classes)
    self.g = g

    print(f"{[type(c) for c in self.classes]}, {self.n_classes}")
    print(self.targets)

    self.samples_per_class = batch_size // self.n_classes
    print(f"samples per class: {self.samples_per_class}")

    # TODO: assert self.target is 1d, batch//   

    # targets has to be a np.array for this to be safe, otherwise is wrong
    self.class_indices = {
      c: np.nonzero(self.targets == c)[0]
      for c in self.classes
    }
    # print(self.class_indices)

     # Minority class size
    self.minority_size = min(
      len(v) for v in self.class_indices.values()
    )
    # identify minority class
    self.minority_class = min(
      self.class_indices,
      key=lambda c: len(self.class_indices[c])
    )
    self.minority_indices = (
      self.class_indices[self.minority_class]
    )
    # print([self.class_indices[3]])
    print(f"minority size: {self.minority_size}")
    print(f"minority class: {self.minority_class}")

    # TODO: fix that to not drop the remaining samples from minority class
    self.n_batches = self.minority_size // self.samples_per_class

    print(f"batches: {self.n_batches}")

  def __iter__(self):

    shuffled = {} # same as class_indices , but shuffled
    for c, idxs in self.class_indices.items():
      perm = torch.randperm(len(idxs), generator=self.g)
      shuffled[c] = idxs.copy()[perm]
      print(f"{c} : {idxs[:6]} | {shuffled[c][:6]}")

    for batch_idx in range(self.n_batches):
      batch = []

      start = batch_idx*self.samples_per_class
      end = start + self.samples_per_class

      for c in self.classes:
        print(f"for class {c} added {shuffled[c][start:end]}")
        batch.extend(shuffled[c][start:end].tolist())

      yield batch

  def __len__(self):
    return self.n_batches

In [5]:
generator = torch.Generator().manual_seed(123456789)
sampler = BalancedBatchSampler(targets=dev_ds.targets, batch_size=24,g=generator)

# print(next(iter(sampler)))
# print(len(next(iter(sampler))))

for batch in islice(sampler, 3):
  print(f"batch: {len(batch)}")

[<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>], 6
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

ValueError: Calling nonzero on 0d arrays is not allowed. Use np.atleast_1d(scalar).nonzero() instead. If the context of this error is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.

In [ ]:
# to 230 diaireitai me 23, 46, 115

list

In [27]:
dev_loader = DataLoader(dev_ds, batch_sampler=sampler, num_workers=4)
# images, labels = next(iter(loader))

for (images, labels) in islice(dev_loader, 2):
  print(images.shape, labels.shape)

0 : [0 1 2 3 4 5] | [23273 37922  5637 34235 17229  6155]
1 : [46836 46837 46838 46839 46840 46841] | [55324 65836 56808 64606 72514 71781]
2 : [73702 73703 73704 73705 73706 73707] | [ 99190  86458  83406 102007  85304  83119]
3 : [102744 102745 102746 102747 102748 102749] | [102765 103056 103014 103124 102902 102946]
4 : [103220 103221 103222 103223 103224 103225] | [103310 103364 103344 103254 103448 103429]
5 : [103450 103451 103452 103453 103454 103455] | [103791 103762 103963 103526 103886 103625]
for class 0 added [23273 37922  5637 34235]
for class 1 added [55324 65836 56808 64606]
for class 2 added [ 99190  86458  83406 102007]
for class 3 added [102765 103056 103014 103124]
for class 4 added [103310 103364 103344 103254]
for class 5 added [103791 103762 103963 103526]
for class 0 added [17229  6155 31456 33033]
for class 1 added [72514 71781 67465 69145]
for class 2 added [85304 83119 84459 84089]
for class 3 added [102902 102946 103102 102935]
for class 4 added [103448 1034

In [ ]:
class BalancedBatchSampler(Sampler):
  """
  Creates batches with equal number of samples per class.

  Example:
      batch_size = 252
      num_classes = 6
      -> 42 samples per class per batch
  """

  def __init__(self, labels, batch_size, shuffle=True):
    self.labels = np.array(labels)
    self.classes = np.unique(self.labels)

    self.num_classes = len(self.classes)

    assert batch_size % self.num_classes == 0, (
      "Batch size must be divisible by number of classes"
    )

    self.samples_per_class = batch_size // self.num_classes
    self.batch_size = batch_size
    self.shuffle = shuffle

    # Store indices for each class
    self.class_indices = {
      c: np.where(self.labels == c)[0].tolist()
      for c in self.classes
    }

    # Minority class determines epoch length
    self.min_class_count = min(len(v) for v in self.class_indices.values())

    # Number of batches so minority class is fully used
    self.num_batches = (
      self.min_class_count // self.samples_per_class
    )

  def __iter__(self):

    # Shuffle class indices every epoch
    pools = {}

    for c, idxs in self.class_indices.items():
      idxs = idxs.copy()

      if self.shuffle:
        random.shuffle(idxs)

      pools[c] = idxs

    class_ptrs = {c: 0 for c in self.classes}

    for _ in range(self.num_batches):

      batch = []

      for c in self.classes:

        idxs = pools[c]
        ptr = class_ptrs[c]

        # If not enough samples remain:
        if ptr + self.samples_per_class > len(idxs):

          # reshuffle and restart majority classes
          if self.shuffle:
            random.shuffle(idxs)

          ptr = 0

        selected = idxs[ptr:ptr+self.samples_per_class]

        batch.extend(selected)

        class_ptrs[c] = ptr + self.samples_per_class

      if self.shuffle:
        random.shuffle(batch)

      yield batch

  def __len__(self):
    return self.num_batches

In [ ]:

ROOT = Path("../data/processed/classification").resolve()

transforms = v2.Compose([
  v2.ToImage(), 
  v2.Resize((64,64)),
])
train_ds = ImageFolder(ROOT / "train", transform=transforms)
dev_ds = ImageFolder(ROOT / "dev", transform=transforms)

print(len(train_ds.targets))

sampler = BalancedBatchSampler(
    labels=train_ds.targets,
    batch_size=252
)

loader = DataLoader(
  dev_ds,
  batch_sampler=sampler,
  num_workers=4
)

for (imgs, targets) in loader:
  print(imgs.shape)

936118
